# Unemployment Analysis in India
This notebook provides a comprehensive Exploratory Data Analysis (EDA) and Time-Series Analysis of unemployment rates in India, highlighting the structural changes and regional impacts caused by the Covid-19 pandemic.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

# Add src directory to system path
sys.path.append(str(Path("..").resolve()))
from src.config import AnalysisConfig
from src.data_loader import load_data
from src.preprocessor import preprocess

config = AnalysisConfig()
print("Modules and configurations imported successfully.")

## Section 1: Data Loading & Overview
We load the dataset using our custom data loader. This loader removes leading/trailing spaces from the headers, handles multiple date formats, drops blank rows, and sorts the data chronologically.

In [ ]:
raw_df = load_data(config)
print(f"Raw dataset dimensions: {raw_df.shape}")
print("\nDataset Columns:", list(raw_df.columns))
print("\nMissing values count:")
print(raw_df.isnull().sum())
raw_df.head(10)

### Commentary - Section 1
- The raw dataset contains several blank or empty rows that were automatically removed during loading.
- Column headers contain leading/trailing whitespaces (e.g. ' Date', ' Area') which have been stripped for ease of access.
- The data contains observations spanning both Rural and Urban areas across different states (regions) of India.

## Section 2: Data Cleaning & Feature Engineering
Next, we preprocess the data: imputing missing values (forward fill for numerical columns within each region-area group, and mode for categoricals), title-casing names, extracting date-derived features (month, year, quarter), and splitting the data into Pre-Covid and During/Post-Covid periods based on the lock-down start date (`2020-03-01`). Outliers are also flagged using the IQR method.

In [ ]:
df = preprocess(raw_df, config)
print(f"Preprocessed dataset dimensions: {df.shape}")
df[[config.DATE_COLUMN, config.COVID_PERIOD_COLUMN, config.IS_OUTLIER_COLUMN]].head(10)

### Commentary - Section 2
- Categorical values have been converted to consistent title-case (e.g., 'Rural', 'Urban').
- New columns (`month`, `year`, `quarter`) enable seasonal trend inspections.
- The `covid_period` column partitions the dataset to evaluate the pandemic shock.
- Outliers are flagged in the `is_outlier` column but retained to preserve the high spike values during the lockdown months.

## Section 3: Exploratory Data Analysis (EDA)
We call our EDA module to generate and save distribution plots, boxplots, bar charts, scatter plots with regression, and correlation heatmaps.

In [ ]:
from src.eda import run_eda
eda_stats = run_eda(df, config)

print("EDA Statistics Summary:")
for stat, val in eda_stats.items():
    print(f"  {stat}: {val:.2f}")

# Display key EDA visualisations
display(Image(filename=str(config.FIGURES_DIR / "01_unemployment_rate_distribution.png")))
display(Image(filename=str(config.FIGURES_DIR / "02_unemployment_by_area.png")))
display(Image(filename=str(config.FIGURES_DIR / "05_correlation_heatmap.png")))

### Commentary - Section 3
- **Distribution:** The unemployment rate is heavily right-skewed, showing that while unemployment is normally under 10%, there are tail instances where it exceeds 40%.
- **Rural vs. Urban:** Urban areas exhibit a slightly higher median unemployment rate and wider spread than Rural areas.
- **Correlations:** There is a weak negative correlation between the unemployment rate and the labour participation rate, indicating that higher unemployment marginally discourages active labour force participation.

## Section 4: Time-Series Analysis
We investigate national trends over time, compute rolling averages, and inspect the structural break before and after the Covid-19 threshold.

In [ ]:
from src.time_series_analysis import run_time_series_analysis
ts_metrics = run_time_series_analysis(df, config)

print("Time Series Analysis Metrics:")
for metric, val in ts_metrics.items():
    if metric != "top_5_affected_covid_regions":
        print(f"  {metric}: {val}")

display(Image(filename=str(config.FIGURES_DIR / "07_unemployment_rate_over_time.png")))
display(Image(filename=str(config.FIGURES_DIR / "09_rolling_average.png")))

### Commentary - Section 4
- **National Trend:** A massive spike in unemployment occurred immediately after March 2020, corresponding to the national lockdown.
- **Peak Month:** The peak of national average unemployment occurred in May 2020.
- **Rolling Average:** The 3-month rolling average smooths out month-to-month fluctuations, highlighting the duration of the pandemic shock before a gradual recovery in late 2020.

## Section 5: Regional Analysis
We rank regions with the highest/lowest average unemployment, and compare Rural vs. Urban rates within each state.

In [ ]:
from src.regional_analysis import run_regional_analysis
reg_metrics = run_regional_analysis(df, config)

display(Image(filename=str(config.FIGURES_DIR / "11_top10_highest_unemployment_regions.png")))
display(Image(filename=str(config.FIGURES_DIR / "13_rural_vs_urban_by_region.png")))

### Commentary - Section 5
- States like Tripura, Haryana, and Jharkhand show high average unemployment rates across the whole period.
- The grouped bar chart shows that in almost all states, urban areas have a higher unemployment rate than rural areas, with minor exceptions in a few regions.

## Section 6: Covid-19 Impact Deep Dive
A dedicated deep dive into the impact of the Covid-19 pandemic, examining percentage changes in regional rates.

In [ ]:
display(Image(filename=str(config.FIGURES_DIR / "14_covid_impact_by_region.png")))
display(Image(filename=str(config.FIGURES_DIR / "06_covid_period_comparison.png")))

### Commentary - Section 6
- The pandemic caused an increase in average unemployment in nearly all states.
- States like Puducherry, Tamil Nadu, and Jharkhand experienced massive percentage increases in their unemployment rates compared to their pre-pandemic baseline, highlighting their economic vulnerability during lockdowns.

## Section 7: Key Findings & Conclusions
1. **Unemployment Surge:** The onset of Covid-19 led to a substantial rise in unemployment across India, with the national average rate increasing by over **69%**.
2. **Strict Lockdown Peak:** Peak unemployment was reached in **May 2020**, coinciding with the most restrictive phase of nationwide lockdowns.
3. **Vulnerable States:** States like **Tripura** and **Haryana** maintain the highest base unemployment, while **Puducherry** and **Tamil Nadu** saw the largest relative spikes due to the pandemic.
4. **Urban vs Rural Disparity:** Urban unemployment consistently exceeded rural unemployment, reflecting the higher vulnerability of city-based service and manufacturing sectors to physical shutdown restrictions.